In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 2.4 MB/s eta 0:00:00


# Configuração Inicial e Classe do Agente

In [2]:
import os
import re
from groq import Groq

os.environ['GROQ_API_KEY'] = "Minha Chave de API"

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

class Agent: # Molde para criar um agente
  def __init__(self, client, system):
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    completion = self.client.chat.completions.create(
        messages=self.messages,
        model="llama-3.3-70b-versatile",
    )
    return completion.choices[0].message.content

# Ferramentas de Gestão de Estoque e Precificação de Loja

In [3]:
def calculate(operation):
    return eval(operation)

def get_product_price(product) -> float: # Retorna o preço unitário do produto em Reais (R$).
    match product.lower():
        case "notebook":
            return 3500.00
        case "smartphone":
            return 1200.50
        case "monitor":
            return 850.00
        case "teclado":
            return 150.00
        case _:
            return 0.0

def get_inventory_count(product) -> int: # Retorna a quantidade de itens no estoque.
    match product.lower():
        case "notebook":
            return 15
        case "smartphone":
            return 40
        case "monitor":
            return 22
        case "teclado":
            return 60
        case _:
            return 0

# Prompt do Sistema

In [4]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_product_price:
e.g. get_product_price: notebook
returns the price of the product

get_inventory_count:
e.g. get_inventory_count: notebook
returns the quantity of the product in stock

Example session:

Question: What is the total inventory value for keyboards (teclado)?
Thought: I need to find the price of a teclado and how many are in stock, then multiply them. Let's get the price first.
Action: get_product_price: teclado
PAUSE

You will be called again with this:

Observation: 150.0

Thought: Now I need to find how many teclados are in stock.
Action: get_inventory_count: teclado
PAUSE

You will be called again with this:

Observation: 60

Thought: Now I need to multiply the price (150.0) by the quantity (60).
Action: calculate: 150.0 * 60
PAUSE

You will be called again with this:

Observation: 9000.0

If you have the answer, output it as the Answer.

Answer: The total inventory value for teclados is R$ 9000.0.

Now it's your turn:
""".strip()

# Loop Autônomo de Execução

In [5]:
def agent_loop(max_iterations, system, query):
  agent = Agent(client, system)
  tools = ['calculate', 'get_product_price', 'get_inventory_count']
  next_prompt = query
  i = 0

  print(f"User: {query}\n")

  while i < max_iterations:
    i += 1
    result = agent(next_prompt)
    print(result)

    if "PAUSE" in result and "Action" in result:
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)

      if action:
          chosen_tool = action[0][0].strip()
          arg = action[0][1].strip()

          if chosen_tool in tools:
            result_tool = eval(f"{chosen_tool}('{arg}')")
            next_prompt = f"Observation: {result_tool}"
          else:
            next_prompt = "Observation: Tool not found"

          print(f"\n---\n{next_prompt}\n---\n")
          continue

    if "Answer" in result:
      break

# Rodando o Agente

In [6]:
pergunta = "What is the total value of all notebooks and monitors in our inventory?"
agent_loop(max_iterations=15, system=system_prompt, query=pergunta)

User: What is the total value of all notebooks and monitors in our inventory?

Thought: To find the total value of all notebooks and monitors in our inventory, we need to calculate the total value of each item separately and then add them together. First, let's find the price and quantity of notebooks. 

Action: get_product_price: notebook
PAUSE

---
Observation: 3500.0
---

Thought: Now that we have the price of a notebook, we need to find out how many notebooks are in stock. 

Action: get_inventory_count: notebook
PAUSE

---
Observation: 15
---

Thought: Now we know the price and quantity of notebooks. Next, we'll calculate the total value of the notebooks in stock. Then, we'll find the price and quantity of monitors and calculate their total value. After that, we can add the total values of notebooks and monitors together.

Action: calculate: 3500.0 * 15
PAUSE

---
Observation: 52500.0
---

Thought: Now that we have the total value of notebooks, let's find the price of a monitor.

A